# Natura 2000 Bronze → Silver

Reads Natura 2000 GeoJSON from S3/local, handles CRS, and writes **two tables**:

| Table | S3 path |
|-------|---------|
| **silver_protected_site** | `s3://ie-datalake/silver/nature2000/protected_site/country=XX/snapshot_date=YYYY-MM-DD/` |
| **silver_protected_site_h3** | `s3://ie-datalake/silver/nature2000/protected_site_h3/snapshot_date=YYYY-MM-DD/` |

### A) silver_protected_site (1 row per SITE_CODE; lookup)
- SITE_CODE, SITE_NAME, site_name_norm, AC, TIPO
- HECTAREAS, area_km2, perimeter_km, compactness, n_vertices, perimeter_area_ratio
- bbox_min/max_lon/lat, bbox_area_km2
- centroid_lon, centroid_lat, centroid_in_bbox_ok
- geometry_type, is_multipolygon, is_valid
- h3_6, h3_7, h3_8, h3_9 (centroid-based)
- crs_original, _source_file, ingested_at_utc, area_source_gap_pct

### B) silver_protected_site_h3 (exploded mapping)
- SITE_CODE, h3_res, h3_id
- overlap_area_km2, cell_area_km2, overlap_fraction
- is_boundary_cell (overlap < 0.999), is_core_cell (>= 0.95)
- rank_by_overlap, site_cover_fraction, is_derived (res 8/9 estimated from parent if overlap ≥ 50%)

In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

S3_BUCKET = "ie-datalake"
BRONZE_PREFIX = "bronze/nature2000"
SILVER_PREFIX = "silver/nature2000"
AWS_PROFILE = "486717354268_PowerUserAccess"

# Bronze GeoJSON files (Spain: Mac = Canaries, Medalpatl = mainland + Balearic)
BRONZE_FILES = [
    "Es_Lic_SCI_Zepa_SPA_Mac_202412.geojson",
    "Es_Lic_SCI_Zepa_SPA_Medalpatl_202412.geojson",
]

# Snapshot date (hardcoded per requirements)
SNAPSHOT_DATE = "2026-02-27"

# H3 resolutions for centroid-based columns (finest first; parents derived from h3_9)
H3_RESOLUTIONS = [9, 8, 7, 6]

# H3 resolutions for polyfill: only 6 and 7 computed; 8 and 9 derived from parents
H3_POLYFILL_RESOLUTIONS = [6, 7]
# Threshold for deriving children: parent overlap >= this → add child with overlap=1.0; else skip
H3_DERIVE_THRESHOLD = 0.5

# CRS for area/perimeter (equal-area; EPSG:3035 = ETRS89-LAEA for Europe)
AREA_CRS = "EPSG:3035"

# Site code column (case-insensitive match: SITE_CODE, SITECODE, etc.)
SITE_CODE_COL = "SITE_CODE"

# If CRS is missing and coords don't look like degrees: use this CRS (e.g. "EPSG:25830" for Spain UTM)
# Set to None to fail loudly. Spain Natura 2000 often uses EPSG:25830.
ASSUME_CRS_IF_MISSING = "EPSG:25830"

# Parquet write settings
PARQUET_COMPRESSION = "snappy"
PARQUET_ROW_GROUP_SIZE = 50_000

# Output table prefixes (S3)
SILVER_PROTECTED_SITE_PREFIX = "silver/nature2000/protected_site"
SILVER_PROTECTED_SITE_H3_PREFIX = "silver/nature2000/protected_site_h3"

In [ ]:
%pip install -q geopandas h3 pyarrow s3fs pyproj

In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# BRONZE → SILVER PIPELINE (CRS-aware, centroid + polyfill, geometry features)
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations

import io
import logging
import math
import time
from datetime import datetime, timezone

import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
from geopandas.io.arrow import _geopandas_to_arrow
from pyproj import Transformer
from shapely.ops import transform

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("nature2000_silver")

fs = s3fs.S3FileSystem(profile=AWS_PROFILE)
log.info("S3FileSystem ready (profile=%s)", AWS_PROFILE)

# Equal-area CRS for area/perimeter (Europe)
_TRANSFORMER_AREA = Transformer.from_crs("EPSG:4326", AREA_CRS, always_xy=True)


def _find_col(gdf: gpd.GeoDataFrame, *names: str) -> str | None:
    """Case-insensitive column lookup."""
    cols_lower = {c.lower(): c for c in gdf.columns}
    for n in names:
        if n.lower() in cols_lower:
            return cols_lower[n.lower()]
    return None


# ══════════════════════════════════════════════════════════════════════════════
# 1. READ GeoJSON from S3 (preserve CRS from file)
# ══════════════════════════════════════════════════════════════════════════════

def read_geojson(path: str) -> gpd.GeoDataFrame:
    """Read GeoJSON from S3 (s3://bucket/key) or local path. Preserve CRS from file."""
    if path.startswith("s3://"):
        parts = path.replace("s3://", "").split("/", 1)
        s3_full = f"{parts[0]}/{parts[1]}" if len(parts) > 1 else parts[0]
        log.info("Reading GeoJSON: %s", path)
        with fs.open(s3_full, "rb") as f:
            gdf = gpd.read_file(io.BytesIO(f.read()))
    else:
        log.info("Reading GeoJSON: %s", path)
        gdf = gpd.read_file(path)
    log.info("Read %d features, %d columns, CRS=%s", len(gdf), len(gdf.columns), gdf.crs)
    return gdf


def read_geojson_from_s3(s3_path: str) -> gpd.GeoDataFrame:
    """Read GeoJSON from S3 (bucket/key format, no s3:// prefix)."""
    log.info("Reading GeoJSON: s3://%s", s3_path)
    with fs.open(s3_path, "rb") as f:
        gdf = gpd.read_file(io.BytesIO(f.read()))
    log.info("Read %d features, %d columns, CRS=%s", len(gdf), len(gdf.columns), gdf.crs)
    return gdf


def _coords_look_like_degrees(gdf: gpd.GeoDataFrame) -> bool:
    """Check if first centroid looks like degrees (lat -90..90, lon -180..180)."""
    geom = gdf.geometry.iloc[0]
    if geom is None or geom.is_empty:
        return False
    pt = geom.centroid
    lon, lat = float(pt.x), float(pt.y)
    return -90 <= lat <= 90 and -180 <= lon <= 180


def ensure_crs_and_reproject_to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Ensure GeoDataFrame has CRS. If missing, infer: assume EPSG:4326 ONLY if
    coordinates look like degrees; otherwise fail loudly.
    Reproject to EPSG:4326 before centroid/H3.
    """
    gdf = gdf.copy()
    crs_orig = str(gdf.crs) if gdf.crs is not None else None

    if gdf.crs is None:
        if _coords_look_like_degrees(gdf):
            log.warning("CRS missing; coordinates look like degrees – assuming EPSG:4326")
            gdf = gdf.set_crs("EPSG:4326")
            crs_orig = "missing (assumed EPSG:4326)"
        elif ASSUME_CRS_IF_MISSING:
            log.warning("CRS missing; coords not degrees – using ASSUME_CRS_IF_MISSING=%s", ASSUME_CRS_IF_MISSING)
            gdf = gdf.set_crs(ASSUME_CRS_IF_MISSING)
            crs_orig = f"missing (assumed {ASSUME_CRS_IF_MISSING})"
        else:
            sample = gdf.geometry.iloc[0]
            pt = sample.centroid if sample else None
            coords = f"x={pt.x:.2f}, y={pt.y:.2f}" if pt else "N/A"
            raise ValueError(
                f"CRS is missing and coordinates do NOT look like degrees. "
                f"Sample centroid: {coords}. Set ASSUME_CRS_IF_MISSING or assign CRS."
            )

    if gdf.crs != "EPSG:4326":
        log.info("Reprojecting from %s to EPSG:4326", gdf.crs)
        gdf = gdf.to_crs("EPSG:4326")

    gdf["crs_original"] = crs_orig
    return gdf


def load_all_bronze() -> gpd.GeoDataFrame:
    """Load and concatenate all bronze GeoJSON files."""
    gdfs = []
    for fname in BRONZE_FILES:
        s3_path = f"{S3_BUCKET}/{BRONZE_PREFIX}/{fname}"
        if not fs.exists(s3_path):
            log.warning("Bronze file not found: s3://%s – skipping", s3_path)
            continue
        gdf = read_geojson_from_s3(s3_path)
        gdf = ensure_crs_and_reproject_to_wgs84(gdf)
        gdf["_source_file"] = fname
        gdfs.append(gdf)
    if not gdfs:
        raise FileNotFoundError(f"No bronze files found. Expected: {BRONZE_FILES}")
    combined = pd.concat(gdfs, ignore_index=True)
    return gpd.GeoDataFrame(combined, geometry="geometry", crs="EPSG:4326")


# ══════════════════════════════════════════════════════════════════════════════
# 2. FIX INVALID GEOMETRIES + DATA QUALITY
# ══════════════════════════════════════════════════════════════════════════════

def fix_invalid_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Fix invalid geometries with buffer(0). Log count."""
    invalid = ~gdf.geometry.is_valid & gdf.geometry.notna()
    n = invalid.sum()
    if n > 0:
        log.info("Fixing %d invalid geometries with buffer(0)", n)
        gdf = gdf.copy()
        gdf.loc[invalid, "geometry"] = gdf.loc[invalid, "geometry"].buffer(0)
    return gdf


# ══════════════════════════════════════════════════════════════════════════════
# 3. EXTRACT ATTRIBUTES + COUNTRY
# ══════════════════════════════════════════════════════════════════════════════

def extract_country_from_sitecode(sitecode) -> str:
    if sitecode is not None and len(str(sitecode).strip()) >= 2:
        return str(sitecode).strip()[:2].upper()
    return "ES"


def extract_attributes(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    sitecode_col = _find_col(gdf, "SITE_CODE", "SITECODE")
    if "country" not in gdf.columns:
        if sitecode_col:
            gdf["country"] = gdf[sitecode_col].apply(extract_country_from_sitecode)
        else:
            gdf["country"] = "ES"
    return gdf


# ══════════════════════════════════════════════════════════════════════════════
# 4. GEOMETRY-DERIVED FEATURES (area_km2, perimeter_km, bbox, compactness, n_vertices)
# ══════════════════════════════════════════════════════════════════════════════

def add_geometry_features(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Add area_km2, perimeter_km, bbox, compactness, n_vertices. Use equal-area CRS for area/perimeter."""
    gdf = gdf.copy()

    def _area_km2(geom):
        if geom is None or geom.is_empty:
            return 0.0
        try:
            g = transform(lambda x, y: _TRANSFORMER_AREA.transform(x, y), geom)
            return float(g.area) / 1_000_000  # m² → km²
        except Exception:
            return 0.0

    def _perimeter_km(geom):
        if geom is None or geom.is_empty:
            return 0.0
        try:
            g = transform(lambda x, y: _TRANSFORMER_AREA.transform(x, y), geom)
            return float(g.length) / 1000  # m → km
        except Exception:
            return 0.0

    gdf["area_km2"] = gdf.geometry.apply(_area_km2)
    gdf["perimeter_km"] = gdf.geometry.apply(_perimeter_km)

    # Bbox in EPSG:4326 (already in WGS84)
    def _bbox(geom):
        if geom is None or geom.is_empty:
            return np.nan, np.nan, np.nan, np.nan
        b = geom.bounds
        return b[0], b[1], b[2], b[3]  # minx, miny, maxx, maxy

    bbox_arr = np.array(gdf.geometry.apply(_bbox).tolist())
    gdf["bbox_min_lon"] = bbox_arr[:, 0]
    gdf["bbox_min_lat"] = bbox_arr[:, 1]
    gdf["bbox_max_lon"] = bbox_arr[:, 2]
    gdf["bbox_max_lat"] = bbox_arr[:, 3]

    # Compactness = 4*pi*area / perimeter^2
    gdf["compactness"] = np.where(
        gdf["perimeter_km"] > 0,
        4 * math.pi * gdf["area_km2"] / (gdf["perimeter_km"] ** 2),
        np.nan,
    )

    # n_vertices (boundary coordinates)
    def _n_vertices(geom):
        if geom is None or geom.is_empty:
            return 0
        if geom.geom_type == "Point":
            return 1
        if hasattr(geom, "exterior"):
            return len(geom.exterior.coords)
        if hasattr(geom, "geoms"):
            return sum(len(p.exterior.coords) for p in geom.geoms)
        return 0

    gdf["n_vertices"] = gdf.geometry.apply(_n_vertices)

    # geometry_type, is_valid
    gdf["geometry_type"] = gdf.geometry.geom_type
    gdf["is_valid"] = gdf.geometry.is_valid

    gdf["ingested_at_utc"] = datetime.now(timezone.utc).isoformat()
    return gdf


# ══════════════════════════════════════════════════════════════════════════════
# 4b. ADD perimeter_area_ratio, bbox_area_km2, centroid_in_bbox_ok, area_source_gap_pct
# ══════════════════════════════════════════════════════════════════════════════

def _normalize_site_name(s: str) -> str:
    """Lowercase, strip, collapse spaces. Optionally remove accents."""
    if pd.isna(s) or s is None:
        return ""
    import unicodedata
    s = str(s).lower().strip()
    s = " ".join(s.split())
    # Remove accents (NFD → filter combining chars)
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    return s


def add_silver_protected_site_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Add columns for silver_protected_site: site_name_norm, perimeter_area_ratio, bbox_area_km2, centroid_in_bbox_ok, is_multipolygon, area_source_gap_pct."""
    gdf = gdf.copy()
    site_name_col = _find_col(gdf, "SITE_NAME", "SITENAME")
    if site_name_col:
        gdf["site_name_norm"] = gdf[site_name_col].apply(_normalize_site_name)
    else:
        gdf["site_name_norm"] = ""

    gdf["perimeter_area_ratio"] = gdf["perimeter_km"] / np.maximum(gdf["area_km2"], 1e-9)

    # bbox_area_km2: bbox polygon in EPSG:4326 → project to EPSG:3035 → area/1e6
    def _bbox_area_km2(row):
        minlon, minlat = row["bbox_min_lon"], row["bbox_min_lat"]
        maxlon, maxlat = row["bbox_max_lon"], row["bbox_max_lat"]
        if np.isnan(minlon) or np.isnan(minlat):
            return np.nan
        from shapely.geometry import box
        bbox = box(minlon, minlat, maxlon, maxlat)
        try:
            g = transform(lambda x, y: _TRANSFORMER_AREA.transform(x, y), bbox)
            return float(g.area) / 1_000_000
        except Exception:
            return np.nan

    gdf["bbox_area_km2"] = gdf.apply(_bbox_area_km2, axis=1)

    # centroid_in_bbox_ok
    gdf["centroid_in_bbox_ok"] = (
        (gdf["centroid_lon"] >= gdf["bbox_min_lon"]) & (gdf["centroid_lon"] <= gdf["bbox_max_lon"])
        & (gdf["centroid_lat"] >= gdf["bbox_min_lat"]) & (gdf["centroid_lat"] <= gdf["bbox_max_lat"])
    )

    gdf["is_multipolygon"] = gdf["geometry_type"].str.contains("Multi", na=False)

    # area_source_gap_pct = (area_km2 - HECTAREAS/100) / max(area_km2, 1e-9)
    hect_col = _find_col(gdf, "HECTAREAS", "HECTARES")
    if hect_col:
        hect_km2 = pd.to_numeric(gdf[hect_col], errors="coerce").fillna(0) / 100
        gdf["area_source_gap_pct"] = (gdf["area_km2"] - hect_km2) / np.maximum(gdf["area_km2"], 1e-9)
    else:
        gdf["area_source_gap_pct"] = np.nan

    return gdf


# ══════════════════════════════════════════════════════════════════════════════
# 5. CENTROID + H3 (must be in EPSG:4326)
# ══════════════════════════════════════════════════════════════════════════════

def add_centroid_and_h3(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    centroid_lat = geometry.centroid.y, centroid_lon = geometry.centroid.x (EPSG:4326).
    Validate ranges; compute h3_6..h3_9 from centroid.
    """
    gdf = gdf.copy()
    def _centroid_lon(geom):
        if geom is None or geom.is_empty:
            return np.nan
        try:
            return float(geom.centroid.x)
        except Exception:
            return np.nan
    def _centroid_lat(geom):
        if geom is None or geom.is_empty:
            return np.nan
        try:
            return float(geom.centroid.y)
        except Exception:
            return np.nan
    gdf["centroid_lon"] = gdf.geometry.apply(_centroid_lon)
    gdf["centroid_lat"] = gdf.geometry.apply(_centroid_lat)

    # Validate (only for non-null)
    valid_mask = gdf["centroid_lat"].notna() & gdf["centroid_lon"].notna()
    bad_lat = valid_mask & ((gdf["centroid_lat"] < -90) | (gdf["centroid_lat"] > 90))
    bad_lon = valid_mask & ((gdf["centroid_lon"] < -180) | (gdf["centroid_lon"] > 180))
    if bad_lat.any() or bad_lon.any():
        idx = (bad_lat | bad_lon).idxmax()
        raise ValueError(
            f"Centroid out of range. CRS={gdf.crs}. "
            f"Example row {idx}: lat={gdf.loc[idx, 'centroid_lat']}, lon={gdf.loc[idx, 'centroid_lon']}"
        )

    # H3 from centroid
    _h3_latlng = np.vectorize(h3.latlng_to_cell, otypes=[str])
    _h3_parent = np.vectorize(h3.cell_to_parent, otypes=[str])
    lat = gdf["centroid_lat"].to_numpy(dtype=float)
    lon = gdf["centroid_lon"].to_numpy(dtype=float)
    valid = np.isfinite(lat) & np.isfinite(lon)
    resolutions = sorted(H3_RESOLUTIONS, reverse=True)
    finest = resolutions[0]
    h9 = np.full(len(gdf), None, dtype=object)
    h9[valid] = _h3_latlng(lat[valid], lon[valid], finest)
    gdf[f"h3_{finest}"] = h9
    for res in resolutions[1:]:
        child_col = f"h3_{res + 1}"
        parent_vals = np.full(len(gdf), None, dtype=object)
        has_child = gdf[child_col].notna()
        if has_child.any():
            parent_vals[has_child] = _h3_parent(gdf.loc[has_child, child_col].to_numpy(), res)
        gdf[f"h3_{res}"] = parent_vals
    log.info("H3 centroid columns: h3_6, h3_7, h3_8, h3_9")
    return gdf


# ══════════════════════════════════════════════════════════════════════════════
# 6. BUILD silver_protected_site (1 row per SITE_CODE)
# ══════════════════════════════════════════════════════════════════════════════

SILVER_SITE_COLUMNS = [
    "SITE_CODE", "SITE_NAME", "site_name_norm", "AC", "TIPO",
    "HECTAREAS", "area_km2", "perimeter_km", "compactness", "n_vertices", "perimeter_area_ratio",
    "bbox_min_lon", "bbox_min_lat", "bbox_max_lon", "bbox_max_lat", "bbox_area_km2",
    "centroid_lon", "centroid_lat", "centroid_in_bbox_ok",
    "geometry_type", "is_multipolygon", "is_valid",
    "h3_6", "h3_7", "h3_8", "h3_9",
    "crs_original", "_source_file", "ingested_at_utc",
    "area_source_gap_pct", "country",
]


def build_silver_protected_site(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Build silver_protected_site: 1 row per SITE_CODE with required columns."""
    sitecode_col = _find_col(gdf, "SITE_CODE", "SITECODE")
    if not sitecode_col:
        raise ValueError("SITE_CODE column required")
    gdf = gdf.drop_duplicates(subset=[sitecode_col], keep="first").copy()
    renames = {}
    if sitecode_col != "SITE_CODE":
        renames[sitecode_col] = "SITE_CODE"
    for old, new in [("SITE_NAME", "SITENAME"), ("HECTAREAS", "HECTARES")]:
        c = _find_col(gdf, old, new)
        if c and c != old:
            renames[c] = old
    if renames:
        gdf = gdf.rename(columns=renames)
    cols = [c for c in SILVER_SITE_COLUMNS if c in gdf.columns]
    out = gdf[cols].copy()
    if "geometry" in gdf.columns and "geometry" not in out.columns:
        out["geometry"] = gdf["geometry"].values
    return gpd.GeoDataFrame(out, geometry="geometry", crs=gdf.crs)


# ══════════════════════════════════════════════════════════════════════════════
# 7. BUILD silver_protected_site_h3 (polyfill + overlap metrics)
# ══════════════════════════════════════════════════════════════════════════════

def _h3_cell_to_polygon_wgs84(h3_id: str):
    """H3 cell boundary as Shapely polygon in WGS84 (lon, lat)."""
    from shapely.geometry import Polygon
    b = h3.cell_to_boundary(h3_id)
    coords = [(lon, lat) for lat, lon in b]
    return Polygon(coords)


def build_silver_protected_site_h3(gdf: gpd.GeoDataFrame, site_area_lookup: pd.Series) -> pd.DataFrame:
    """
    Build silver_protected_site_h3: SITE_CODE, h3_res, h3_id, overlap_area_km2, cell_area_km2,
    overlap_fraction, is_boundary_cell, is_core_cell, rank_by_overlap, site_cover_fraction, is_derived.
    Res 6–7: polyfill + intersection. Res 8–9: derived from parent if overlap_fraction >= 50%.
    """
    sitecode_col = _find_col(gdf, "SITE_CODE", "SITECODE")
    if not sitecode_col:
        return pd.DataFrame()

    site_geom_col = "geometry" if "geometry" in gdf.columns else gdf.geometry.name
    rows = []
    n_sites = len(gdf)
    for i, (idx, row) in enumerate(gdf.iterrows()):
        if (i + 1) % 200 == 0:
            log.info("  site_h3 progress: %d/%d", i + 1, n_sites)
        geom = row[site_geom_col]
        if geom is None or geom.is_empty:
            continue
        site_code = str(row[sitecode_col]) if pd.notna(row[sitecode_col]) else f"idx_{idx}"
        site_area_km2 = site_area_lookup.get(site_code, row.get("area_km2", np.nan))
        if pd.isna(site_area_km2):
            site_area_km2 = 0.0
        try:
            geom_3035 = transform(lambda x, y: _TRANSFORMER_AREA.transform(x, y), geom)
        except Exception:
            continue
        geo = geom.__geo_interface__
        for res in H3_POLYFILL_RESOLUTIONS:
            try:
                cells = h3.geo_to_cells(geo, res)
            except Exception:
                continue
            for h3_id in cells:
                cell_area_m2 = h3.cell_area(h3_id, unit="m^2")
                cell_area_km2 = cell_area_m2 / 1_000_000
                hex_poly = _h3_cell_to_polygon_wgs84(h3_id)
                try:
                    hex_3035 = transform(lambda x, y: _TRANSFORMER_AREA.transform(x, y), hex_poly)
                    inter = geom_3035.intersection(hex_3035)
                    overlap_m2 = inter.area if not (inter.is_empty or inter is None) else 0.0
                except Exception:
                    overlap_m2 = 0.0
                overlap_km2 = overlap_m2 / 1_000_000
                overlap_frac = overlap_km2 / cell_area_km2 if cell_area_km2 > 0 else 0.0
                site_cover_frac = overlap_km2 / site_area_km2 if site_area_km2 > 0 else np.nan
                rows.append({
                    "SITE_CODE": site_code, "h3_res": res, "h3_id": h3_id,
                    "overlap_area_km2": overlap_km2, "cell_area_km2": cell_area_km2,
                    "overlap_fraction": overlap_frac,
                    "is_boundary_cell": overlap_frac < 0.999,
                    "is_core_cell": overlap_frac >= 0.95,
                    "site_cover_fraction": site_cover_frac,
                    "is_derived": False,
                })

    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)

    # Derive res 8 and 9 from parents: parent overlap >= 50% → add children with overlap=1.0
    def _add_derived_rows(d: pd.DataFrame, parent_res: int, child_res: int) -> pd.DataFrame:
        parents = d[(d["h3_res"] == parent_res) & (d["overlap_fraction"] >= H3_DERIVE_THRESHOLD)]
        if parents.empty:
            return d
        derived = []
        for _, r in parents.iterrows():
            for h3_id in h3.cell_to_children(r["h3_id"], child_res):
                cell_area_km2 = h3.cell_area(h3_id, unit="m^2") / 1_000_000
                site_area_km2 = site_area_lookup.get(r["SITE_CODE"], np.nan) or 0.0
                site_cover_frac = cell_area_km2 / site_area_km2 if site_area_km2 > 0 else np.nan
                derived.append({
                    "SITE_CODE": r["SITE_CODE"], "h3_res": child_res, "h3_id": h3_id,
                    "overlap_area_km2": cell_area_km2, "cell_area_km2": cell_area_km2,
                    "overlap_fraction": 1.0,
                    "is_boundary_cell": False, "is_core_cell": True,
                    "site_cover_fraction": site_cover_frac, "is_derived": True,
                })
        if derived:
            return pd.concat([d, pd.DataFrame(derived)], ignore_index=True)
        return d

    df = _add_derived_rows(df, 7, 8)
    df = _add_derived_rows(df, 8, 9)

    df["rank_by_overlap"] = df.groupby(["SITE_CODE", "h3_res"])["overlap_fraction"].rank(ascending=False, method="first")
    log.info("silver_protected_site_h3: %d rows (%d sites, res 6,7 computed; 8,9 derived)", len(df), df["SITE_CODE"].nunique())
    return df


# ══════════════════════════════════════════════════════════════════════════════
# 8. WRITE
# ══════════════════════════════════════════════════════════════════════════════

def _table_no_dictionary(tbl: pa.Table) -> pa.Table:
    for i in range(tbl.num_columns):
        if pa.types.is_dictionary(tbl.schema.field(i).type):
            tbl = tbl.set_column(i, tbl.schema.field(i).name, tbl.column(i).dictionary_decode())
    return tbl


def write_silver(site_df: gpd.GeoDataFrame, site_h3_df: pd.DataFrame) -> dict[str, str]:
    """Write silver_protected_site and silver_protected_site_h3 to S3."""
    site_df = site_df.copy()
    site_df["snapshot_date"] = SNAPSHOT_DATE
    s3_site = f"s3://{S3_BUCKET}/{SILVER_PROTECTED_SITE_PREFIX}"
    log.info("Writing silver_protected_site: %d rows to %s", len(site_df), s3_site)
    tbl = _geopandas_to_arrow(site_df, index=False, geometry_encoding="WKB")
    tbl = _table_no_dictionary(tbl)
    pq.write_to_dataset(
        tbl, root_path=s3_site, filesystem=fs,
        partition_cols=["country", "snapshot_date"],
        existing_data_behavior="delete_matching",
        row_group_size=min(PARQUET_ROW_GROUP_SIZE, len(tbl)),
        compression=PARQUET_COMPRESSION, use_dictionary=False,
    )
    if len(site_h3_df) > 0:
        site_h3_df = site_h3_df.copy()
        site_h3_df["snapshot_date"] = SNAPSHOT_DATE
        s3_h3 = f"s3://{S3_BUCKET}/{SILVER_PROTECTED_SITE_H3_PREFIX}"
        log.info("Writing silver_protected_site_h3: %d rows to %s", len(site_h3_df), s3_h3)
        pq.write_to_dataset(
            pa.Table.from_pandas(site_h3_df, preserve_index=False),
            root_path=s3_h3, filesystem=fs,
            partition_cols=["snapshot_date"],
            existing_data_behavior="delete_matching",
            compression=PARQUET_COMPRESSION, use_dictionary=False,
        )
    countries = site_df["country"].unique().tolist()
    return {c: f"{s3_site}/country={c}/snapshot_date={SNAPSHOT_DATE}" for c in countries}

00:10:06 [INFO] S3FileSystem ready (profile=486717354268_PowerUserAccess)


In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN PIPELINE – run cells 0–3 first, then execute this cell
# ─────────────────────────────────────────────────────────────────────────────

t_start = time.time()

gdf = load_all_bronze()
gdf = fix_invalid_geometries(gdf)
gdf = extract_attributes(gdf)
gdf = add_geometry_features(gdf)
gdf = add_centroid_and_h3(gdf)
gdf = add_silver_protected_site_columns(gdf)

site_df = build_silver_protected_site(gdf)
site_area_lookup = site_df.set_index("SITE_CODE")["area_km2"]
site_h3_df = build_silver_protected_site_h3(gdf, site_area_lookup)
written = write_silver(site_df, site_h3_df)

elapsed = time.time() - t_start
log.info("✓ Pipeline complete in %.0fs: %d sites → silver_protected_site, %d rows → silver_protected_site_h3", elapsed, len(site_df), len(site_h3_df))
for country, uri in written.items():
    log.info("  %s: %s", country, uri)

00:10:10 [INFO] Reading GeoJSON: s3://ie-datalake/bronze/nature2000/Es_Lic_SCI_Zepa_SPA_Mac_202412.geojson
00:10:18 [INFO] Read 226 features, 9 columns, CRS=EPSG:32628
00:10:18 [INFO] Reprojecting from EPSG:32628 to EPSG:4326
00:10:18 [INFO] Reading GeoJSON: s3://ie-datalake/bronze/nature2000/Es_Lic_SCI_Zepa_SPA_Medalpatl_202412.geojson
00:11:35 [INFO] Read 1636 features, 9 columns, CRS=EPSG:25830
00:11:35 [INFO] Reprojecting from EPSG:25830 to EPSG:4326
00:11:38 [INFO] Fixing 12 invalid geometries with buffer(0)
00:12:24 [INFO] H3 centroid columns: h3_6, h3_7, h3_8, h3_9
00:12:32 [INFO]   site_h3 progress: 200/1862
00:13:23 [INFO]   site_h3 progress: 400/1862
00:13:31 [INFO]   site_h3 progress: 600/1862
00:14:01 [INFO]   site_h3 progress: 800/1862
00:14:10 [INFO]   site_h3 progress: 1000/1862
00:14:15 [INFO]   site_h3 progress: 1200/1862
00:14:20 [INFO]   site_h3 progress: 1400/1862
00:15:29 [INFO]   site_h3 progress: 1600/1862
00:15:46 [INFO]   site_h3 progress: 1800/1862
00:16:21 [I

In [ ]:
# Optional: preview silver_protected_site (run after main pipeline)
preview_cols = ["SITE_CODE", "SITE_NAME", "country", "centroid_lat", "centroid_lon", "area_km2"]
preview_cols += [c for c in ["h3_6", "h3_7", "h3_8", "h3_9"] if c in site_df.columns]
preview_cols = [c for c in preview_cols if c in site_df.columns]
display(site_df[preview_cols].head(5))

In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# VERIFY – quick sanity check
# ─────────────────────────────────────────────────────────────────────────────

import pyarrow.dataset as ds

sample_country = list(written.keys())[0] if written else "ES"
s3_site = f"{S3_BUCKET}/{SILVER_PROTECTED_SITE_PREFIX}/country={sample_country}/snapshot_date={SNAPSHOT_DATE}"
print(f"silver_protected_site: s3://{s3_site}")
ds_site = ds.dataset(s3_site, filesystem=fs, format="parquet")
df_site = ds_site.to_table().to_pandas()
print(f"  {len(df_site):,} rows, centroid_lat [{df_site['centroid_lat'].min():.4f}, {df_site['centroid_lat'].max():.4f}]")
display(df_site[[c for c in df_site.columns if c != "geometry"][:14]].head(3))

s3_h3 = f"{S3_BUCKET}/{SILVER_PROTECTED_SITE_H3_PREFIX}/snapshot_date={SNAPSHOT_DATE}"
print(f"silver_protected_site_h3: s3://{s3_h3}")
ds_h3 = ds.dataset(s3_h3, filesystem=fs, format="parquet")
df_h3 = ds_h3.to_table().to_pandas()
print(f"  {len(df_h3):,} rows")
display(df_h3.head(5))

silver_protected_site: s3://ie-datalake/silver/nature2000/protected_site/country=ES/snapshot_date=2026-02-27
  1,862 rows, centroid_lat [25.6449, 44.0425]


,SITE_CODE,SITE_NAME,site_name_norm,AC,TIPO,HECTAREAS,area_km2,perimeter_km,compactness,n_vertices,perimeter_area_ratio,bbox_min_lon,bbox_min_lat,bbox_max_lon
0,ES0000108,Los Órganos,los organos,CANARIAS,B,152.404571,1.523401,10.938751,0.159988,1762,7.180480,-17.279019,28.207222,-17.256998
1,ES0000111,Tamadaba,tamadaba,CANARIAS,B,7488.701273,74.937499,58.304478,0.277016,6332,0.778041,-15.822997,27.986217,-15.660969
2,ES7010002,Barranco Oscuro,barranco oscuro,CANARIAS,B,33.482255,0.335062,4.124717,0.247484,271,12.310293,-15.597420,28.058697,-15.586768


silver_protected_site_h3: s3://ie-datalake/silver/nature2000/protected_site_h3/snapshot_date=2026-02-27
  3,728,374 rows


,SITE_CODE,h3_res,h3_id,overlap_area_km2,cell_area_km2,overlap_fraction,is_boundary_cell,is_core_cell,site_cover_fraction,is_derived,rank_by_overlap
0,ES0000111,6,86344cb17ffffff,29.727196,41.349192,0.718931,True,False,0.396693,False,1.0
1,ES0000111,6,86344cbafffffff,21.640638,41.345257,0.523413,True,False,0.288782,False,2.0
2,ES0000111,7,87344cba1ffffff,5.899501,5.908472,0.998482,True,True,0.078726,False,3.0
3,ES0000111,7,87344cba3ffffff,4.991267,5.909189,0.844662,True,False,0.066606,False,8.0
4,ES0000111,7,87344cba5ffffff,2.614259,5.909111,0.442411,True,False,0.034886,False,10.0
